# SGN Tutorial 02: Modern Dataset Track

This notebook validates the modern dataset curation contract for SGN tutorials.
It is designed for deterministic headless execution in CI and local `uv run` workflows.


In [ ]:
from pathlib import Path

try:
    import tomllib
except ModuleNotFoundError:  # pragma: no cover - Python < 3.11 fallback
    import tomli as tomllib

repo_root = Path.cwd()
manifest_path = repo_root / "docs" / "tutorials" / "dataset_manifest.toml"
with manifest_path.open("rb") as handle:
    manifest = tomllib.load(handle)

manifest["manifest"]

In [ ]:
species = manifest["species"]
rows = []
for species_key, species_cfg in species.items():
    for run_accession in species_cfg["run_accessions"]:
        run_meta = species_cfg["run_metadata"][run_accession]
        rows.append(
            {
                "species_key": species_key,
                "run_accession": run_accession,
                "group": run_meta["group"],
                "layout": run_meta["library_layout"],
            }
        )

rows

In [ ]:
supported = set(manifest["manifest"]["supported_acquisition"])
required = {"python-httpx", "sra-toolkit"}
assert required.issubset(supported), (required, supported)

print("supported acquisition modes:", sorted(supported))

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

group_counts = {}
for row in rows:
    group_counts[row["group"]] = group_counts.get(row["group"], 0) + 1

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(list(group_counts.keys()), list(group_counts.values()), color="#264653")
ax.set_title("Modern tutorial run groups")
ax.set_ylabel("Run count")
fig.tight_layout()
fig